# Day 09 — RAG Setup with pgvector

## Objectives

- Understand vector databases
- Connect Python to PostgreSQL
- Verify pgvector
- Create an embeddings table
- Store sample embeddings
- Run nearest-neighbour similarity search
- Create an HNSW index
- Inspect query behaviour
- Prepare the final project ERD

In [1]:
import psycopg
from pgvector.psycopg import register_vector
from pgvector import Vector

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="genai_api",
    user="postgres",
    password="postgres"
)

register_vector(conn)

print("Connected to PostgreSQL successfully")

Connected to PostgreSQL successfully


In [3]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT extname, extversion
        FROM pg_extension
        WHERE extname = 'vector';
    """)
    
    result = cur.fetchone()

print("pgvector:", result)

pgvector: ('vector', '0.8.6')


In [4]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = 'embeddings'
        ORDER BY ordinal_position;
    """)
    
    columns = cur.fetchall()

for column in columns:
    print(column)

('id', 'integer')
('content', 'text')
('embedding', 'USER-DEFINED')
('metadata', 'jsonb')


In [5]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT id, content, embedding, metadata
        FROM embeddings
        ORDER BY id;
    """)
    
    rows = cur.fetchall()

for row in rows:
    print(row)

(1, 'Employees receive 18 days of annual leave.', Vector([0.10000000149011612, 0.20000000298023224, 0.30000001192092896, 0.4000000059604645]), {'page': 12, 'source': 'employee_handbook.pdf', 'category': 'leave'})
(2, 'Employees receive 12 days of sick leave.', Vector([0.11999999731779099, 0.2199999988079071, 0.2800000011920929, 0.3799999952316284]), {'page': 13, 'source': 'employee_handbook.pdf', 'category': 'leave'})
(3, 'The office working hours are 9 AM to 6 PM.', Vector([0.800000011920929, 0.699999988079071, 0.6000000238418579, 0.5]), {'page': 5, 'source': 'company_policy.pdf', 'category': 'office'})
(4, 'Employees can work remotely two days per week.', Vector([0.75, 0.6499999761581421, 0.550000011920929, 0.44999998807907104]), {'page': 3, 'source': 'remote_policy.pdf', 'category': 'remote'})


In [6]:
query_vector = Vector([0.11, 0.21, 0.29, 0.39])

In [7]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            id,
            content,
            embedding <=> %s AS distance,
            metadata
        FROM embeddings
        ORDER BY embedding <=> %s
        LIMIT 3;
    """, (query_vector, query_vector))
    
    results = cur.fetchall()

for row in results:
    print(row)

(1, 'Employees receive 18 days of annual leave.', 0.0005929313477044396, {'page': 12, 'source': 'employee_handbook.pdf', 'category': 'leave'})
(2, 'Employees receive 12 days of sick leave.', 0.000622942072493804, {'page': 13, 'source': 'employee_handbook.pdf', 'category': 'leave'})
(3, 'The office working hours are 9 AM to 6 PM.', 0.15321407725024427, {'page': 5, 'source': 'company_policy.pdf', 'category': 'office'})


In [8]:
for row in results:
    print(f"ID: {row[0]}")
    print(f"Content: {row[1]}")
    print(f"Distance: {row[2]:.6f}")
    print(f"Metadata: {row[3]}")
    print("-" * 60)

ID: 1
Content: Employees receive 18 days of annual leave.
Distance: 0.000593
Metadata: {'page': 12, 'source': 'employee_handbook.pdf', 'category': 'leave'}
------------------------------------------------------------
ID: 2
Content: Employees receive 12 days of sick leave.
Distance: 0.000623
Metadata: {'page': 13, 'source': 'employee_handbook.pdf', 'category': 'leave'}
------------------------------------------------------------
ID: 3
Content: The office working hours are 9 AM to 6 PM.
Distance: 0.153214
Metadata: {'page': 5, 'source': 'company_policy.pdf', 'category': 'office'}
------------------------------------------------------------


## Similarity Search Observation

The query vector represents a hypothetical user question.

The `<=>` operator calculates cosine distance between the query
embedding and stored embeddings.

The query sorts results by ascending distance.

Therefore:

- Smaller distance = more similar
- Larger distance = less similar
- LIMIT 3 returns the three nearest vectors

In [10]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT indexname, indexdef
        FROM pg_indexes
        WHERE tablename = 'embeddings';
    """)
    
    indexes = cur.fetchall()

for index in indexes:
    print(index)

('embeddings_pkey', 'CREATE UNIQUE INDEX embeddings_pkey ON public.embeddings USING btree (id)')
('embeddings_embedding_hnsw_idx', 'CREATE INDEX embeddings_embedding_hnsw_idx ON public.embeddings USING hnsw (embedding vector_cosine_ops)')


In [11]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    id,
    content,
    embedding <=> %s AS distance
FROM embeddings
ORDER BY embedding <=> %s
LIMIT 3;
"""

with conn.cursor() as cur:
    cur.execute(query, (query_vector, query_vector))
    plan = cur.fetchall()

for line in plan:
    print(line[0])

Limit  (cost=1.10..1.11 rows=3 width=52) (actual time=0.037..0.039 rows=3 loops=1)
  Buffers: shared hit=1
  ->  Sort  (cost=1.10..1.11 rows=4 width=52) (actual time=0.034..0.035 rows=3 loops=1)
        Sort Key: ((embedding <=> '[0.11,0.21,0.29,0.39]'::vector))
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1
        ->  Seq Scan on embeddings  (cost=0.00..1.06 rows=4 width=52) (actual time=0.020..0.023 rows=4 loops=1)
              Buffers: shared hit=1
Planning:
  Buffers: shared hit=1
Planning Time: 0.127 ms
Execution Time: 0.070 ms


## HNSW Index

HNSW stands for Hierarchical Navigable Small World.

It is an approximate nearest-neighbour indexing method used to
speed up vector similarity searches.

The index used in this project is:

embeddings_embedding_hnsw_idx

It uses:

vector_cosine_ops

The HNSW index becomes more useful as the number of stored
embeddings increases.

With only four sample vectors, PostgreSQL may choose a sequential
scan because it can be cheaper than using the index.

In [12]:
def similarity_search(query_vector, top_k=3):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                id,
                content,
                embedding <=> %s AS distance,
                metadata
            FROM embeddings
            ORDER BY embedding <=> %s
            LIMIT %s;
        """, (query_vector, query_vector, top_k))
        
        return cur.fetchall()

In [13]:
results = similarity_search(
    Vector([0.11, 0.21, 0.29, 0.39]),
    top_k=3
)

for result in results:
    print(result)

(1, 'Employees receive 18 days of annual leave.', 0.0005929313477044396, {'page': 12, 'source': 'employee_handbook.pdf', 'category': 'leave'})
(2, 'Employees receive 12 days of sick leave.', 0.000622942072493804, {'page': 13, 'source': 'employee_handbook.pdf', 'category': 'leave'})
(3, 'The office working hours are 9 AM to 6 PM.', 0.15321407725024427, {'page': 5, 'source': 'company_policy.pdf', 'category': 'office'})
